In [ ]:
# Lab type: review
# Course: DS105 — Exploratory Data Analysis
# Lesson: Relationships Between Variables
# Task: Audit the AI-generated correlation analysis below. The code is correct and runs without errors.
#       The written commentary contains two problems:
#         (1) one causal claim that the data does not support
#         (2) one correlation described as a 'strong, consistent linear relationship'
#             that a scatter plot will contradict
#       Your tasks:
#         1. Find the causal claim and rewrite it as a correct correlational statement
#         2. Add the scatter plot that tests whether the flagged correlation is actually linear
#         3. Check whether the key relationship holds consistently across channels

## About This Lab

An AI tool produced the correlation analysis below for the orders dataset.
All code cells are correct — run them in order to reproduce the output.

Your job is to evaluate the **written commentary** in each `> AI-generated summary` block,
identify where the claims go beyond what the data supports, and add the visualisations
the AI skipped.

## Setup: Build the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(7)
n = 83600

channel = np.random.choice(
    ['web', 'mobile', 'in-store', 'phone'],
    size=n,
    p=[0.45, 0.35, 0.15, 0.05]
)

quantity = np.random.choice([1, 2, 3, 4, 5], n, p=[0.4, 0.3, 0.15, 0.1, 0.05]).astype(float)
bulk_idx = np.random.choice(n, 18, replace=False)
quantity[bulk_idx] = np.random.randint(50, 501, 18)

unit_price = np.round(
    np.random.choice([18.99, 29.99, 39.99, 49.99, 69.99, 99.99], n), 2
)

# discount: higher probability for lower-priced items (matches real-world promo patterns)
disc_prob = np.where(unit_price <= 29.99, 0.30, np.where(unit_price <= 49.99, 0.20, 0.10))
apply_discount = np.random.random(n) < disc_prob
discount = np.zeros(n)
discount[apply_discount] = np.round(np.random.uniform(0.05, 0.30, apply_discount.sum()), 2)

# shipping_cost: correlated with quantity
shipping_cost = np.round(
    2.99 + quantity * 1.50 + np.random.normal(0, 2.0, n), 2
)
shipping_cost = np.clip(shipping_cost, 1.99, None)

# revenue: quantity x unit_price x (1 - discount) x noise
revenue = np.round(
    quantity * unit_price * (1 - discount) * np.random.uniform(0.88, 1.04, n), 2
)

df = pd.DataFrame({
    'order_id':      ['ORD-{:06d}'.format(i) for i in range(1, n + 1)],
    'channel':       channel,
    'quantity':      quantity,
    'unit_price':    unit_price,
    'discount':      discount,
    'shipping_cost': shipping_cost,
    'revenue':       revenue,
})

print(f'Shape: {df.shape}')
print(df[['quantity', 'unit_price', 'discount', 'shipping_cost', 'revenue']].describe().round(2))

---

## Step 1: Correlation Matrix

In [ ]:
corr = df[['quantity', 'unit_price', 'discount', 'shipping_cost', 'revenue']].corr()
print(corr.round(2))

> **AI-generated summary:**
>
> The correlation matrix reveals several key relationships in the orders dataset.
> `quantity` and `revenue` are positively correlated (r ≈ 0.61), which is expected — larger
> orders generate more revenue. The strongest predictor of revenue is `unit_price` (r ≈ 0.74),
> reflecting that higher-priced products generate more revenue per order.
>
> `shipping_cost` shows moderate positive correlation with both `quantity` (r ≈ 0.38) and
> `revenue` (r ≈ 0.51), consistent with larger, higher-value orders incurring higher shipping costs.
>
> **The negative correlation between `discount` and `revenue` (r ≈ −0.18) confirms that the
> discounting strategy is driving down revenue. The data shows that reducing or eliminating
> discounts would improve revenue performance.**
>
> `unit_price` and `discount` show a small negative correlation (r ≈ −0.08), indicating that
> discounts are applied more frequently to lower-priced items.

---

### Task 1 of 3 — Identify the causal claim

The AI summary above contains one passage that uses causal language without causal evidence.
A correlation coefficient tells you that two variables move together — it says nothing about
which (if either) causes the other.

1. Quote the specific sentence(s) that make a causal claim.
2. Rewrite them as a correct correlational statement.
3. Briefly explain what alternative explanation(s) could account for the observed correlation
   without discounts causing lower revenue.

*Your answer here.*

---

## Step 2: Correlation Heatmap

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation matrix — orders dataset')
plt.tight_layout()
plt.show()

> **AI-generated summary:**
>
> The heatmap makes the correlation structure easy to scan. The `unit_price`–`revenue`
> cell (r ≈ 0.74) is the strongest off-diagonal value, confirming a **strong, consistent
> linear relationship**: revenue increases at a steady, predictable rate as unit prices rise
> across the entire price range. **No further visualisation is needed to interpret this
> relationship** — the correlation coefficient captures it fully.

---

## Step 3: Add the Missing Scatter Plot

The AI commentary describes the `unit_price`–`revenue` relationship as
"strong, consistent, and linear" and concludes that no further visualisation is needed.

Pearson correlation measures **linear** association only. A high r does not mean:
- the relationship is uniformly strong across the full range of the x-variable
- the variance in y is constant as x increases
- there are no structural sub-groups the number is averaging over

Add a scatter plot of `unit_price` (x-axis) vs `revenue` (y-axis) in the cell below.

In [ ]:
# Task 2 of 3: add a scatter plot of unit_price vs revenue
# Use alpha=0.15 to handle overplotting at the lower price points


**After running your scatter plot, answer the following:**

1. Does the spread of revenue values look roughly the same at every price point,
   or does it fan out as unit price increases?
2. What does this tell you about the AI's claim of a "steady, predictable rate across
   the entire price range"?
3. A linear regression model fitted to this data would use a single slope to describe
   the unit_price–revenue relationship. Based on what you see in the scatter, what
   limitation would that model have at high price points?

*Your answer here.*

---

## Step 4: Does the Relationship Hold Across Channels?

The correlation matrix was computed on the full dataset. A relationship that looks
strong overall may differ across sub-groups — and a relationship that reverses within
a segment is invisible in the aggregate number.

Reproduce your scatter plot from Step 3 with `channel` encoded as colour.

In [ ]:
# Task 3 of 3: reproduce the scatter plot with hue='channel'
# Move the legend outside the plot area: legend=False then plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')


**After running the channel-coloured scatter, answer the following:**

1. Does the `unit_price`–`revenue` relationship look roughly the same across all channels,
   or does one channel show a markedly different pattern?
2. If you were building a model to predict `revenue` and found that the relationship differs
   meaningfully by channel, what would that imply about whether `channel` belongs in the model?
3. The AI-generated correlation analysis ran on the full dataset and reported a single
   r value. What is the risk of relying on that number without segmenting by channel first?

*Your answer here.*